In [43]:
from multi_dim_engine import Tensor
from optimizer import Adam
import numpy as np

In [44]:

def cross_entropy_loss(index_vector,scores) :
    # we get the probabilities matrix
    probabilities = softmax(scores)

    #set up for the advanced indexing
    rows = np.arange(probabilities.data.shape[0])
    true_prob = Tensor(probabilities.data[rows, index_vector])

    #calculating the losses 
    losses = Tensor(-np.log(true_prob.data + 1e-5))
    N = true_prob.data.shape[0]
    
    full_loss = Tensor(losses.data.mean() ,(scores,))

    def _backward() :
        num_classes = probabilities.data.shape[1]
        Y = np.eye(num_classes)[index_vector]
        scores.grad = (probabilities.data - Y)/N
        
    
    full_loss._backward = _backward
        
    return full_loss

def softmax(scores) :
    shifted_scores = scores.data - np.max(scores.data , axis = -1 , keepdims=True)
    expon = np.exp(shifted_scores)
    prob = expon / np.sum(expon, axis = -1, keepdims=True)
    return Tensor(prob)
    

class Layer :

    def __init__(self, n_inputs , n_neurons) :
        self.w = Tensor(np.random.rand(n_neurons,n_inputs))
        self.b = Tensor(np.random.rand(n_neurons))

    def __call__(self,x) :
        output = x @ self.w.T() + self.b
        return output

    def parameters(self) :
        return [self.w , self.b]
    

    
class MLP : 

    def __init__(self, n_inputs,n_outputs) :
        size = [n_inputs] + n_outputs 
        self.layers = [Layer(size[i], size[i+1]) for i in range(len(size)-1)]

    def __call__(self,x) :
        for layer in self.layers[:-1] : 
            x = layer(x).relu()
        scores = self.layers[-1](x)
        return scores

    def parameters(self) :
        params = []
        for layer in self.layers :
            params.extend(layer.parameters())
        return params

In [45]:
np.random.seed(42)

x = Tensor([[1,2,3],
            [4,5,6],
            [7,8,9]])

model = MLP(3,[4,4])
target_vector = [1,1,1]



In [46]:
optimizer = Adam(model.parameters(), 0.01)

for i in range(400) :
    scores = model(x)
    full_loss = cross_entropy_loss(target_vector,scores)

    optimizer.zero_grad()
    full_loss.backward()

    optimizer.step()
    if (i+1) % 50 == 0 or i == 0 :
        print(f"Loss in iteration {i+1} = {full_loss.data}")
    


Loss in iteration 1 = 2.0419131735963143
Loss in iteration 50 = 0.0042901278640640585
Loss in iteration 100 = 0.002322360457379217
Loss in iteration 150 = 0.0014032453875216847
Loss in iteration 200 = 0.0008821227949496574
Loss in iteration 250 = 0.0005752681392898769
Loss in iteration 300 = 0.0003861122574640719
Loss in iteration 350 = 0.0002646379116384276
Loss in iteration 400 = 0.00018397470376269018
